# Run the ensemble extractions operationally - on lots of data

We've got the transcription models to be good enough - let's put them to use

In [1]:
# Step 1. Get a big sample of the images in a flat directory
#
# On SCRATCH - or we'll run out of disc space

import subprocess


# Location of main image dataset, already sized and filtered
SOURCE = "/data/scratch/philip.brohan/documents/Daily_Rainfall_UK/jpgs_25pc_filtered"
# Location to save sampled images and transcriptions
OUTPUT = "/data/scratch/philip.brohan/documents/Daily_Rainfall_UK/operational_sample"
# Output directory on Azure ML datastore
AML_OUTPUT = "operational_sample"
# Number of images to sample
COUNT = 50000
# Random seed for reproducibility
SEED = 42


In [2]:

subprocess.run(
    [
        "python",
        "../../scripts/sample_unseen_images.py",
        "--source-root",
        SOURCE,
        "--output-root", 
        OUTPUT,
        "--count",
        str(COUNT),
        "--seed", # Random seed for reproducibility
        str(SEED),
    ],
    check=True,
)


Scanning recursively: /data/scratch/philip.brohan/documents/Daily_Rainfall_UK/jpgs_25pc_filtered
Found images: 634893
Sampled images: 50000
Flat images dir: /data/scratch/philip.brohan/documents/Daily_Rainfall_UK/operational_sample/images
Manifest CSV: /data/scratch/philip.brohan/documents/Daily_Rainfall_UK/operational_sample/sample_manifest.csv
Manifest JSONL: /data/scratch/philip.brohan/documents/Daily_Rainfall_UK/operational_sample/sample_manifest.jsonl


CompletedProcess(args=['python', '../../scripts/sample_unseen_images.py', '--source-root', '/data/scratch/philip.brohan/documents/Daily_Rainfall_UK/jpgs_25pc_filtered', '--output-root', '/data/scratch/philip.brohan/documents/Daily_Rainfall_UK/operational_sample', '--count', '50000', '--seed', '42'], returncode=0)

In [ ]:
# Upload the sampled images to the Azure ML datastore:
# Deleting the existing contents of the datastore directory first, if necessary
subprocess.run(
    [
        "bash",
        "../../scripts/aml_delete.sh",
        AML_OUTPUT,
    ],
    check=True,
)
subprocess.run(
    [
        "bash",
        "../../scripts/aml_upload.sh",
        "--src",
        OUTPUT,
        "--dst",
        AML_OUTPUT,
    ],
    check=True,
)   

# Note, this cell produces 13Mb of output, delete the outputs a.s.a.p - we don't want to fill git up with them.

That's got the images selected and uploaded. Next step is to run an extraction on those images with each of the operational models.

In [4]:
# Basic setup - model names, batch sizes, etc.
# Specify the model checkpoints manually, rather than auto-generating, so we can be clear what we've used.
# Here I've just specified the 2nd order models.

import json
import subprocess
from datetime import datetime, timezone
from pathlib import Path
from posixpath import normpath

# Define model settings for validation jobs.
# Each entry: (label, checkpoint_path, batch_size, total_shards)
MODEL_SETTINGS = [
    ("SmolVLM", "Daily_rainfall_sample/outputs/checkpoints/HuggingFaceTB--SmolVLM2-2.2B-Instruct-20260629-150922/HuggingFaceTB--SmolVLM2-2.2B-Instruct", 50, 1),
    ("Granite4", "Daily_rainfall_sample/outputs/checkpoints/ibm-granite--granite-vision-4.1-4b-20260629-150950/ibm-granite--granite-vision-4.1-4b", 50, 1),
    ("Gemma3", "Daily_rainfall_sample/outputs/checkpoints/google--gemma-3-4b-it-20260629-151007/google--gemma-3-4b-it", 50, 1),
    ("Gemma4", "Daily_rainfall_sample/outputs/checkpoints/google--gemma-4-E4B-it-20260629-151026/google--gemma-4-E4B-it", 50, 1),
    ("Ministral", "Daily_rainfall_sample/outputs/checkpoints/mistralai--Mistral-Small-3.1-24B-Instruct-2503-20260629-151046/mistralai--Mistral-Small-3.1-24B-Instruct-2503", 15, 1),
]

# ND96amsr A100 v4 has 8 GPUs per node. Use one extraction worker per GPU.
NODE_GPU_WORKERS = 8

print(f"Node GPU workers per extraction job: {NODE_GPU_WORKERS}")

Node GPU workers per extraction job: 8


In [5]:
# Now run an extraction job, with each checkpoint, on the image sample.

for model_name, checkpoint, batch_size, total_shards in MODEL_SETTINGS:
    print(f"Submitting {model_name}...")
    subprocess.run(
        [
            "bash",
            "../../scripts/aml_submit.sh",
            "--checkpoint",
            checkpoint,
            "--images-path",
            AML_OUTPUT + "/images",
            "--transcriptions-path",
            AML_OUTPUT + "/transcriptions",
            "--total-shards",
            str(total_shards),
            "--node-gpu-workers",
            str(NODE_GPU_WORKERS),
            "--batch-size",
            str(batch_size),
            "--extraction-registry",
            "../../outputs/extraction_registry.json",
            "extract",
        ],
        check=True,
    )

Submitting SmolVLM...
Workspace:  mlw-llmdatarescue-uksouth-01  (rg-climate-llmdatarescue / 79c7890c-2a30-44ef-aa8d-419d25b7bb8e)
Compute:    A100x8
GPU workers per node: 8
Finetune GPU processes: 8
Grad accum steps (base): 8
Auto-scale grad accum:   true
Model:      smolvlm
Images:     azureml://subscriptions/79c7890c-2a30-44ef-aa8d-419d25b7bb8e/resourcegroups/rg-climate-llmdatarescue/workspaces/mlw-llmdatarescue-uksouth-01/datastores/large_datastore/paths/operational_sample/images
Outputs:    azureml://subscriptions/79c7890c-2a30-44ef-aa8d-419d25b7bb8e/resourcegroups/rg-climate-llmdatarescue/workspaces/mlw-llmdatarescue-uksouth-01/datastores/large_datastore/paths/Daily_rainfall_sample/outputs

Submitting 1 extract shard(s)...
  Checkpoint: azureml://subscriptions/79c7890c-2a30-44ef-aa8d-419d25b7bb8e/resourcegroups/rg-climate-llmdatarescue/workspaces/mlw-llmdatarescue-uksouth-01/datastores/large_datastore/paths/Daily_rainfall_sample/outputs/checkpoints/HuggingFaceTB--SmolVLM2-2.2B-Ins

Class DeploymentTemplateOperations: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.
Class AutoDeleteSettingSchema: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.
Class AutoDeleteConditionSchema: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.
Class BaseAutoDeleteSettingSchema: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.
Class IntellectualPropertySchema: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.
Class ProtectionLevelSchema: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.
Class BaseIntellectualPropertySchema: Th

affable_angle_rb7ll1hqcp
Added extraction registry: ../../outputs/extraction_registry.json
  Run: 20260630-135710
  Model: smolvlm
  Dataset: operational_sample/images
Submitted 1 extract job(s).
Extraction registry: ../../outputs/extraction_registry.json
Submitting Granite4...
Workspace:  mlw-llmdatarescue-uksouth-01  (rg-climate-llmdatarescue / 79c7890c-2a30-44ef-aa8d-419d25b7bb8e)
Compute:    A100x8
GPU workers per node: 8
Finetune GPU processes: 8
Grad accum steps (base): 8
Auto-scale grad accum:   true
Model:      smolvlm
Images:     azureml://subscriptions/79c7890c-2a30-44ef-aa8d-419d25b7bb8e/resourcegroups/rg-climate-llmdatarescue/workspaces/mlw-llmdatarescue-uksouth-01/datastores/large_datastore/paths/operational_sample/images
Outputs:    azureml://subscriptions/79c7890c-2a30-44ef-aa8d-419d25b7bb8e/resourcegroups/rg-climate-llmdatarescue/workspaces/mlw-llmdatarescue-uksouth-01/datastores/large_datastore/paths/Daily_rainfall_sample/outputs

Submitting 1 extract shard(s)...
  Che

Class DeploymentTemplateOperations: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.
Class AutoDeleteSettingSchema: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.
Class AutoDeleteConditionSchema: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.
Class BaseAutoDeleteSettingSchema: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.
Class IntellectualPropertySchema: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.
Class ProtectionLevelSchema: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.
Class BaseIntellectualPropertySchema: Th

happy_papaya_q32spbk7cr
Added extraction registry: ../../outputs/extraction_registry.json
  Run: 20260630-135751
  Model: smolvlm
  Dataset: operational_sample/images
Submitted 1 extract job(s).
Extraction registry: ../../outputs/extraction_registry.json
Submitting Gemma3...
Workspace:  mlw-llmdatarescue-uksouth-01  (rg-climate-llmdatarescue / 79c7890c-2a30-44ef-aa8d-419d25b7bb8e)
Compute:    A100x8
GPU workers per node: 8
Finetune GPU processes: 8
Grad accum steps (base): 8
Auto-scale grad accum:   true
Model:      smolvlm
Images:     azureml://subscriptions/79c7890c-2a30-44ef-aa8d-419d25b7bb8e/resourcegroups/rg-climate-llmdatarescue/workspaces/mlw-llmdatarescue-uksouth-01/datastores/large_datastore/paths/operational_sample/images
Outputs:    azureml://subscriptions/79c7890c-2a30-44ef-aa8d-419d25b7bb8e/resourcegroups/rg-climate-llmdatarescue/workspaces/mlw-llmdatarescue-uksouth-01/datastores/large_datastore/paths/Daily_rainfall_sample/outputs

Submitting 1 extract shard(s)...
  Checkp

...

dynamic_stick_pq9fr2hv34
Added extraction registry: ../../outputs/extraction_registry.json
  Run: 20260630-135809
  Model: smolvlm
  Dataset: operational_sample/images
Submitted 1 extract job(s).
Extraction registry: ../../outputs/extraction_registry.json
Submitting Gemma4...
Workspace:  mlw-llmdatarescue-uksouth-01  (rg-climate-llmdatarescue / 79c7890c-2a30-44ef-aa8d-419d25b7bb8e)
Compute:    A100x8
GPU workers per node: 8
Finetune GPU processes: 8
Grad accum steps (base): 8
Auto-scale grad accum:   true
Model:      smolvlm
Images:     azureml://subscriptions/79c7890c-2a30-44ef-aa8d-419d25b7bb8e/resourcegroups/rg-climate-llmdatarescue/workspaces/mlw-llmdatarescue-uksouth-01/datastores/large_datastore/paths/operational_sample/images
Outputs:    azureml://subscriptions/79c7890c-2a30-44ef-aa8d-419d25b7bb8e/resourcegroups/rg-climate-llmdatarescue/workspaces/mlw-llmdatarescue-uksouth-01/datastores/large_datastore/paths/Daily_rainfall_sample/outputs

Submitting 1 extract shard(s)...
  Check

Class DeploymentTemplateOperations: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.
Class AutoDeleteSettingSchema: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.
Class AutoDeleteConditionSchema: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.
Class BaseAutoDeleteSettingSchema: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.
Class IntellectualPropertySchema: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.
Class ProtectionLevelSchema: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.
Class BaseIntellectualPropertySchema: Th

ivory_pocket_25c31sqs81
Added extraction registry: ../../outputs/extraction_registry.json
  Run: 20260630-135835
  Model: smolvlm
  Dataset: operational_sample/images
Submitted 1 extract job(s).
Extraction registry: ../../outputs/extraction_registry.json
Submitting Ministral...
Workspace:  mlw-llmdatarescue-uksouth-01  (rg-climate-llmdatarescue / 79c7890c-2a30-44ef-aa8d-419d25b7bb8e)
Compute:    A100x8
GPU workers per node: 8
Finetune GPU processes: 8
Grad accum steps (base): 8
Auto-scale grad accum:   true
Model:      smolvlm
Images:     azureml://subscriptions/79c7890c-2a30-44ef-aa8d-419d25b7bb8e/resourcegroups/rg-climate-llmdatarescue/workspaces/mlw-llmdatarescue-uksouth-01/datastores/large_datastore/paths/operational_sample/images
Outputs:    azureml://subscriptions/79c7890c-2a30-44ef-aa8d-419d25b7bb8e/resourcegroups/rg-climate-llmdatarescue/workspaces/mlw-llmdatarescue-uksouth-01/datastores/large_datastore/paths/Daily_rainfall_sample/outputs

Submitting 1 extract shard(s)...
  Che

Class DeploymentTemplateOperations: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.
Class AutoDeleteSettingSchema: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.
Class AutoDeleteConditionSchema: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.
Class BaseAutoDeleteSettingSchema: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.
Class IntellectualPropertySchema: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.
Class ProtectionLevelSchema: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.
Class BaseIntellectualPropertySchema: Th

joyful_calypso_r3sg7mhtcx
Added extraction registry: ../../outputs/extraction_registry.json
  Run: 20260630-135853
  Model: smolvlm
  Dataset: operational_sample/images
Submitted 1 extract job(s).
Extraction registry: ../../outputs/extraction_registry.json


When the jobs have been submitted, the extraction registry will contain run names needed for download and analysis.

Run the next cell to discover run names from the registry, then paste the printed `RUN_NAMES = [...]` block into the following cell.

In [ ]:
# Discover run names from extraction registry after submissions complete.
# This cell does NOT write external files. It prints a block to paste into the next cell.

EXTRACTION_REGISTRY_PATH = Path("../../outputs/extraction_registry.json")
TARGET_IMAGES_PATH = Path(AML_OUTPUT + "/images")

registry = json.loads(EXTRACTION_REGISTRY_PATH.read_text(encoding="utf-8"))
entries = registry.get("extractions", [])


def _norm_rel(path_like: str) -> str:
    return normpath(path_like.replace("\\", "/")).lstrip("./")

def _parse_created_at(value: str) -> datetime:
    if not value:
        return datetime.min.replace(tzinfo=timezone.utc)
    try:
        return datetime.fromisoformat(value.replace("Z", "+00:00"))
    except ValueError:
        return datetime.min.replace(tzinfo=timezone.utc)


RUN_NAMES = []
missing = []
target_images_norm = _norm_rel(str(TARGET_IMAGES_PATH))

for model_name, checkpoint, _batch_size, _total_shards in MODEL_SETTINGS:
    candidates = [
        e
        for e in entries
        if _norm_rel(str(e.get("images_path", ""))) == target_images_norm
        and str(e.get("checkpoint_path", "")) == checkpoint
        and e.get("run_name")
    ]
    if not candidates:
        missing.append(model_name)
        continue
    best = max(
        candidates,
        key=lambda e: _parse_created_at(str(e.get("created_at", ""))),
    )
    run_name = str(best["run_name"])
    RUN_NAMES.append(run_name)
    print(f"{model_name}: {run_name}")

if missing:
    raise RuntimeError(
        "Missing extraction runs in registry for: "
        + ", ".join(missing)
        + ". Run extraction submission first, then re-run this cell."
    )


print("\nCopy this block into the next cell:\n")
print("RUN_NAMES = [")
for run_name in RUN_NAMES:
    print(f'    "{run_name}",')
print("]\n")


SmolVLM: 20260630-135710
Granite4: 20260630-135751
Gemma3: 20260630-135809
Gemma4: 20260630-135835
Ministral: 20260630-135853

Copy this block into the next cell:

RUN_NAMES = [
    "20260630-135710",
    "20260630-135751",
    "20260630-135809",
    "20260630-135835",
    "20260630-135853",
]



In [3]:
# Persistent run names for this notebook.
# Paste the RUN_NAMES block printed by the previous cell here.
RUN_NAMES = [
    "20260630-135710",
    "20260630-135751",
    "20260630-135809",
    "20260630-135835",
    "20260630-135853",
]

if not RUN_NAMES:
    raise RuntimeError("RUN_NAMES is empty. Run the previous cell, then paste its output here.")

print("Using run names:")
for run_name in RUN_NAMES:
    print("  ", run_name)

Using run names:
   20260630-135710
   20260630-135751
   20260630-135809
   20260630-135835
   20260630-135853


In [ ]:
# When the jobs have completed successfully,
#  download the extractions so we can analyze them locally.
for run_name in RUN_NAMES:
    subprocess.run(
        ["bash", "../../scripts/aml_download.sh", "--run-name", run_name, "--output-dir",
          f"{OUTPUT}/individual_transcriptions/{run_name}"],
        check=True,
    )

Assemble the 5 different model transcriptions into one ensemble transcriptions output directory.

This will contain 1 file per image, same format as the regular transcriptions, except that for each cell there is an array of 5 transcriptions instead of a single one. If the transcriptions for an individual model are missing (the extraction failed) they are entered as 'missing' in the array.

This is the production output for the transcription process - an ensemble transcription of all pages, ready for QC and further analysis.

In [4]:
cmd = [
    "python",
    "../../scripts/build_ensemble_transcriptions.py",
    "--output-dir",
    f"{OUTPUT}/ensemble_transcriptions/",
]

for run_name in RUN_NAMES:
    extraction_dir = f"{OUTPUT}/individual_transcriptions/{run_name}/extractions/{run_name}"
    cmd.extend(["--input-dir", extraction_dir])

subprocess.run(cmd, check=True)

{
  "total_stems": 45990,
  "total_cells": 17660160,
  "fully_present_cells": 17337600,
  "partial_cells": 322560,
  "empty_cells": 0,
  "missing_model_files": 0,
  "parse_failed_or_invalid": 896,
  "n_models": 5,
  "input_dirs": [
    "/data/scratch/philip.brohan/documents/Daily_Rainfall_UK/operational_sample/individual_transcriptions/20260630-135710/extractions/20260630-135710",
    "/data/scratch/philip.brohan/documents/Daily_Rainfall_UK/operational_sample/individual_transcriptions/20260630-135751/extractions/20260630-135751",
    "/data/scratch/philip.brohan/documents/Daily_Rainfall_UK/operational_sample/individual_transcriptions/20260630-135809/extractions/20260630-135809",
    "/data/scratch/philip.brohan/documents/Daily_Rainfall_UK/operational_sample/individual_transcriptions/20260630-135835/extractions/20260630-135835",
    "/data/scratch/philip.brohan/documents/Daily_Rainfall_UK/operational_sample/individual_transcriptions/20260630-135853/extractions/20260630-135853"
  ],
  "p

CompletedProcess(args=['python', '../../scripts/build_ensemble_transcriptions.py', '--output-dir', '/data/scratch/philip.brohan/documents/Daily_Rainfall_UK/operational_sample/ensemble_transcriptions/', '--input-dir', '/data/scratch/philip.brohan/documents/Daily_Rainfall_UK/operational_sample/individual_transcriptions/20260630-135710/extractions/20260630-135710', '--input-dir', '/data/scratch/philip.brohan/documents/Daily_Rainfall_UK/operational_sample/individual_transcriptions/20260630-135751/extractions/20260630-135751', '--input-dir', '/data/scratch/philip.brohan/documents/Daily_Rainfall_UK/operational_sample/individual_transcriptions/20260630-135809/extractions/20260630-135809', '--input-dir', '/data/scratch/philip.brohan/documents/Daily_Rainfall_UK/operational_sample/individual_transcriptions/20260630-135835/extractions/20260630-135835', '--input-dir', '/data/scratch/philip.brohan/documents/Daily_Rainfall_UK/operational_sample/individual_transcriptions/20260630-135853/extractions/2

recision": 3,
  "fully_present_fraction": 0.981735,
  "partial_fraction": 0.018265
}
